# FrogNET ID - Colab GPU Runner
This notebook only pulls the repo and calls functions. All real logic lives in the repo, edited via Claude Code.

In [ ]:
# Mount Google Drive once per runtime, from a real notebook cell (has
# live kernel access for the auth flow). Scripts run later via
# '!python -m ...' execute as subprocesses and can't complete this
# handshake themselves - config.py just re-checks it is already mounted.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Uses a fine-grained GitHub PAT (Contents: Read-only, scoped to this repo)
# stored in Colab Secrets as GH_TOKEN - never hardcode it in a cell.
from google.colab import userdata
import os
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')

!git clone https://{os.environ['GH_TOKEN']}@github.com/vvweeks/frog-id.git
%cd frog-id
!pip install -r requirements.txt -q

In [ ]:
# Set secrets for this session (Colab Secrets panel, or set directly - do NOT commit real keys)
from google.colab import userdata
import os
os.environ['XC_API_KEY'] = userdata.get('XC_API_KEY')
os.environ['INAT_CONTACT_EMAIL'] = 'your_email@example.com'

In [ ]:
# Pull latest code changes (after Claude Code has pushed updates)
!git pull

### One-time migration (run once, then delete this cell)
Older runs stored audio in `train/` and `test/` folders split by quality. The pipeline is now manifest-driven with a flat `<species>/` layout, so flatten any existing data and clear the old-format embeddings once. Safe to skip if you have no prior data.

In [ ]:
import os, glob, shutil
DATA='/content/drive/MyDrive/6575_Deep_Learning/frog_id_outputs/ct_frog_dataset'
for split in ['train','test']:
    base=os.path.join(DATA, split)
    if not os.path.isdir(base): continue
    for sp in os.listdir(base):
        dst=os.path.join(DATA, sp); os.makedirs(dst, exist_ok=True)
        for f in glob.glob(os.path.join(base, sp, '*')):
            shutil.move(f, os.path.join(dst, os.path.basename(f)))
    shutil.rmtree(base)
# Old embeddings were single-pooled at 22kHz; delete so build_embeddings
# regenerates them per-segment at 48kHz.
for p in glob.glob(os.path.join(DATA,'birdnet_embeddings','*.npy')): os.remove(p)
print('migration done')

In [ ]:
!python -m scripts.download_data

In [ ]:
# Register any on-disk audio not yet in the manifest (e.g. older Xeno-canto
# files the API couldn't back-fill). Harmless if there's nothing to add.
!python -m scripts.sync_manifest

In [ ]:
# Assign recordist-disjoint, species-stratified train/val/test.
# ~1/folds to test and to val, rest to train (default 7).
!python -m scripts.make_split --folds 7

In [ ]:
!python -m scripts.build_embeddings

In [ ]:
!python -m scripts.run_training --model resnet --nickname "baseline"
# or: !python -m scripts.run_training --model birdnet

### LLM head-to-head comparison
Export the held-out **test recordings** (one prediction per recording, matching the model's test eval) for an LLM to identify, then score its answers against ground truth.

In [ ]:
# Build llm_testset/: anonymized whole test recordings + prompt.txt + answer_key.csv
!python -m scripts.export_llm_testset
# Then: download llm_testset/, upload recordings/ + prompt.txt to the LLM,
# save its reply as responses.csv (clip_id,species), and score it below.

In [ ]:
!python -m scripts.score_llm responses.csv

In [ ]:
# Head-to-head table: ground truth vs FrogNET vs LLM per test recording,
# with per-model accuracy. Use the run_id from results_log.csv.
!python -m scripts.compare_models --model resnet --run-id <RUN_ID> \
    --responses /content/drive/MyDrive/6575_Deep_Learning/frog_id_outputs/responses.csv
# writes model_vs_llm.csv into llm_testset/